# Апта 12 — Үй тапсырмасы (өте детальді түсіндірмемен)

Бұл Jupyter Notebook-та 12-апта тақырыптары **теория + практика + түсіндірме** түрінде берілген:

1) **Классификация метрикалары**  
   - Confusion Matrix (TP, FP, FN, TN)  
   - Accuracy, Precision, Recall, F1-score  
   - ROC curve және ROC-AUC  

2) **Регрессия метрикалары**  
   - MSE, RMSE, MAE  

3) **Кросс-валидация (Stratified K-Fold)**  
   - Неге керек, қалай есептеледі  

4) **Гиперпараметрлерді іріктеу**  
   - GridSearchCV  
   - RandomizedSearchCV  

5) **Теңгерімсіз кластармен жұмыс**  
   - `class_weight='balanced'`  
   - **SMOTE**  

6) **Оптимизация (Gradient Descent)**  
   - Қарапайым 1D мысал: y ≈ w·x + b

> Ескерту: Мысалдар үшін синтетикалық (жасанды) деректер қолданылады.  
> Негізгі мақсат — формулаларды түсіну және sklearn арқылы қолдана алу.


In [ ]:
# ============================================================
# 1) Импорттар (кітапханаларды жүктеу)
# ============================================================
# numpy/pandas: массивтер мен кестелер
import numpy as np
import pandas as pd

# sklearn: модельдер, метрикалар, кросс-валидация
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, RocCurveDisplay,
    mean_squared_error, mean_absolute_error
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification, make_regression

# визуализация
import matplotlib.pyplot as plt

# Нәтижелер тұрақты болуы үшін random seed
np.random.seed(42)


## 2) Дерек дайындау (Classification)

Біз 2 түрлі датасет жасаймыз:

### A) Balanced (теңгерімді)
Кластар үлесі 50% / 50%. Бұл жерде Accuracy көбіне дұрыс жұмыс істейді.

### B) Imbalanced (теңгерімсіз)
Кластар үлесі 95% / 5%.  
Бұл жағдайда Accuracy жиі **алдап** (misleading) көрсетеді, сондықтан Precision/Recall/F1 және ROC-AUC маңыздырақ.


In [ ]:
# ============================================================
# 2A) Balanced дерек (50/50)
# ============================================================
X_bal, y_bal = make_classification(
    n_samples=2000,       # бақылаулар саны
    n_features=10,        # feature саны
    n_informative=6,      # пайдалы feature саны
    n_redundant=2,        # артық (корреляцияланған) feature саны
    n_clusters_per_class=2,
    weights=[0.5, 0.5],   # теңгерімді
    random_state=42
)

# ============================================================
# 2B) Imbalanced дерек (95/5)
# ============================================================
X_imb, y_imb = make_classification(
    n_samples=4000,
    n_features=12,
    n_informative=7,
    n_redundant=2,
    n_clusters_per_class=2,
    weights=[0.95, 0.05], # теңгерімсіз
    flip_y=0.01,          # аздап шум (қате белгі)
    random_state=42
)

# Кластар үлесін көрейік
pd.Series(y_imb).value_counts().rename("Count"), pd.Series(y_imb).value_counts(normalize=True).rename("Share")


## 3) Confusion Matrix және негізгі метрикалар

### Confusion Matrix
Бинарлы классификацияда 4 жағдай бар:

- **TP (True Positive)**: модель positive деді және расымен positive  
- **FP (False Positive)**: модель positive деді, бірақ шын мәнінде negative  
- **FN (False Negative)**: модель negative деді, бірақ шын мәнінде positive  
- **TN (True Negative)**: модель negative деді және расымен negative  

Осы 4 мән арқылы метрикалар есептеледі.

### Accuracy
\$
Accuracy = \frac{TP + TN}{TP + FP + FN + TN}
\$

**Неге кейде жаман?**  
Егер positive өте аз болса, модель бәрін 0 деп болжауы мүмкін → Accuracy жоғары, бірақ пайдалы емес.

### Precision
\$
Precision = \frac{TP}{TP + FP}
\$

**Мағынасы**: «Positive деп болжағандардың қаншасы расымен positive?»  
Жалған positive (FP) қымбат болғанда маңызды (мысалы: спам емес хатты спам деп жіберіп қою).

### Recall (Sensitivity)
\$
Recall = \frac{TP}{TP + FN}
\$

**Мағынасы**: «Нақты positive-тердің қаншасын тауып алдық?»  
Жалған negative (FN) қымбат болғанда маңызды (мысалы: ауруды өткізіп алу).

### F1-score
\$
F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}
\$

Precision мен Recall арасындағы баланс.  
Теңгерімсіз кластарда көбіне жақсы таңдау.

### ROC-AUC
ROC curve: TPR vs FPR.  
AUC = қисық астындағы аудан.  
- 0.5: кездейсоқ  
- 1.0: өте жақсы


In [ ]:
# ============================================================
# 3) Train/Test split (imbalance дерек)
# ============================================================
# stratify=y_imb -> train/test ішінде класс үлесі сақталады
X_train, X_test, y_train, y_test = train_test_split(
    X_imb, y_imb, test_size=0.25, random_state=42, stratify=y_imb
)

# ============================================================
# 3A) Базалық модель: Logistic Regression
# ============================================================
# Pipeline: StandardScaler -> LogisticRegression
# StandardScaler неліктен керек?
# - Логистикалық регрессия градиентпен оқиды
# - Әртүрлі масштабтағы feature-лер оқуды қиындатады
baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, random_state=42))
])

baseline.fit(X_train, y_train)

y_pred = baseline.predict(X_test)                 # 0/1 болжам
y_proba = baseline.predict_proba(X_test)[:, 1]    # positive ықтималдығы (ROC-AUC үшін керек)


In [ ]:
# ============================================================
# 3B) Метрикаларды есептейтін көмекші функция
# ============================================================
def print_metrics(y_true, y_pred, y_proba=None, title=""):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print("="*60)
    print(title)
    print("-"*60)
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    if y_proba is not None:
        auc = roc_auc_score(y_true, y_proba)
        print(f"ROC-AUC  : {auc:.4f}")
    print("="*60)

print_metrics(y_test, y_pred, y_proba, title="Baseline Logistic Regression (imbalanced)")


### Confusion Matrix визуализациясы (Baseline)

Матрицада ең маңыздысы:
- **FN** көп болса → Recall төмен (positive-ті өткізіп алу көп)  
- **FP** көп болса → Precision төмен (жалған дабыл көп)


In [ ]:
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm).plot()
plt.title("Confusion Matrix — Baseline (imbalanced)")
plt.show()


### ROC curve (Baseline)

ROC curve — threshold өзгергенде TPR және FPR қалай өзгеретінін көрсетеді.  
AUC үлкен болған сайын модель кластарды жақсы ажыратады.


In [ ]:
RocCurveDisplay.from_estimator(baseline, X_test, y_test)
plt.title("ROC Curve — Baseline (imbalanced)")
plt.show()


## 4) Accuracy неге теңгерімсіз деректе жаңылыстырады?

Мысал: егер тестте 95% класс 0 болса, модель **бәрін 0** деп айтса да Accuracy ≈ 95% болады.  
Бірақ ол **1 класын мүлде таппайды** → Recall = 0, F1 = 0.

Төменде дәл осыны көрсетеміз.


In [ ]:
y_pred_all_zero = np.zeros_like(y_test)

print_metrics(y_test, y_pred_all_zero, title="Bad baseline: Predict ALL zeros")

cm_bad = confusion_matrix(y_test, y_pred_all_zero)
ConfusionMatrixDisplay(cm_bad).plot()
plt.title("Confusion Matrix — Predict all zeros")
plt.show()


## 5) Imbalanced шешімі 1: `class_weight='balanced'`

Идея: Loss функциясында minority class-қа үлкен салмақ беру.

`class_weight='balanced'` салмақты автоматты есептейді:
- minority көп «қымбат» болады → модель оны көбірек табуға тырысады.

Артықшылықтары:
- Деректерді өзгертпейді (oversampling жоқ)
- Жылдам


In [ ]:
cw_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])

cw_model.fit(X_train, y_train)
y_pred_cw = cw_model.predict(X_test)
y_proba_cw = cw_model.predict_proba(X_test)[:, 1]

print_metrics(y_test, y_pred_cw, y_proba_cw, title="LogReg + class_weight='balanced'")

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_cw)).plot()
plt.title("Confusion Matrix — class_weight='balanced'")
plt.show()


## 6) Imbalanced шешімі 2: SMOTE

SMOTE minority класын **синтетикалық нүктелер** арқылы көбейтеді.

Формула идеясы (интуитивті):
\$
x_{new} = x_i + \lambda(x_{nn} - x_i), \ \lambda \in [0,1]
\$

Яғни minority ішіндегі екі көрші нүктенің арасында жаңа нүкте пайда болады.

> Бұл үшін `imbalanced-learn` керек. Егер Colab-та ашсаңыз, орнату оңай.


In [ ]:
# Егер керек болса:
# !pip -q install imbalanced-learn


In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

smote_model = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, random_state=42))
])

smote_model.fit(X_train, y_train)
y_pred_sm = smote_model.predict(X_test)
y_proba_sm = smote_model.predict_proba(X_test)[:, 1]

print_metrics(y_test, y_pred_sm, y_proba_sm, title="LogReg + SMOTE")

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_sm)).plot()
plt.title("Confusion Matrix — SMOTE")
plt.show()


## 7) Кросс-валидация (Stratified K-Fold)

### Неге керек?
Train/Test split бір рет қана бөлінеді → нәтиже «сәтті/сәтсіз» бөлінуге тәуелді болуы мүмкін.

### K-Fold идеясы
Дерек K бөлікке бөлінеді.  
Әр итерацияда:
- 1 бөлік — validation
- қалған K−1 — training

Соңында метрикаларды орташа аламыз.

### StratifiedKFold
Классификацияда әр fold-та класстар үлесі **сақталуы керек** → stratified.


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model_cv = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])

# F1 және ROC-AUC бойынша CV бағалау
scores_f1 = cross_val_score(model_cv, X_imb, y_imb, scoring="f1", cv=skf)
scores_auc = cross_val_score(model_cv, X_imb, y_imb, scoring="roc_auc", cv=skf)

print("F1 scores:", np.round(scores_f1, 4))
print("Mean F1 :", scores_f1.mean().round(4), "±", scores_f1.std().round(4))

print("\nROC-AUC scores:", np.round(scores_auc, 4))
print("Mean AUC  :", scores_auc.mean().round(4), "±", scores_auc.std().round(4))


## 8) Гиперпараметрлерді іріктеу

### Параметр vs гиперпараметр
- **Параметр** — модель оқыту кезінде үйренетін мәндер (w, b)
- **Гиперпараметр** — оқытуға дейін берілетін баптаулар (max_depth, C, learning_rate)

### GridSearchCV
Берілген тордағы барлық комбинацияны тексереді.  
Артықшылығы: «толық» іздеу  
Кемшілігі: баяу болуы мүмкін

### RandomizedSearchCV
Кездейсоқ комбинацияларды белгілі N рет таңдайды.  
Артықшылығы: жылдам және практикада жиі жақсы.


In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from scipy.stats import loguniform

# ============================================================
# 8A) GridSearch: Decision Tree (balanced дерек)
# ============================================================
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_bal, y_bal, test_size=0.25, random_state=42, stratify=y_bal
)

dt = DecisionTreeClassifier(random_state=42)

param_grid = {
    "max_depth": [2, 3, 4, 5, 6, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10]
}

grid = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

grid.fit(X_train2, y_train2)
print("Best params:", grid.best_params_)
print("Best CV F1 :", round(grid.best_score_, 4))

best_dt = grid.best_estimator_
y_pred_dt = best_dt.predict(X_test2)
print_metrics(y_test2, y_pred_dt, title="DecisionTree (best from GridSearch) — Test metrics")


In [ ]:
# ============================================================
# 8B) RandomizedSearch: Logistic Regression (imbalanced дерек)
# ============================================================
logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, random_state=42))
])

param_dist = {
    "clf__C": loguniform(1e-3, 1e2),   # кең диапазон (лог-скала)
    "clf__solver": ["lbfgs"],
    "clf__penalty": ["l2"]
}

rand = RandomizedSearchCV(
    estimator=logreg,
    param_distributions=param_dist,
    n_iter=20,
    scoring="roc_auc",
    cv=5,
    random_state=42,
    n_jobs=-1
)

rand.fit(X_train, y_train)
print("Best params:", rand.best_params_)
print("Best CV AUC:", round(rand.best_score_, 4))

best_lr = rand.best_estimator_
y_pred_lr = best_lr.predict(X_test)
y_proba_lr = best_lr.predict_proba(X_test)[:, 1]
print_metrics(y_test, y_pred_lr, y_proba_lr, title="LogReg (best from RandomizedSearch) — Test metrics")


## 9) Регрессия метрикалары (MSE, RMSE, MAE)

Регрессияда мақсат — санды болжау.

- **MSE**: квадратталған қателердің орташа мәні  
  Үлкен қателерге қатты «айыппұл» береді.  
- **RMSE**: MSE түбірі  
  Target өлшем бірлігімен бірдей → түсіну жеңіл.  
- **MAE**: абсолют қателердің орташа мәні  
  Аутлайерлерге төзімді.


In [ ]:
# Регрессияға синтетикалық дерек жасаймыз
Xr, yr = make_regression(n_samples=1500, n_features=6, noise=15.0, random_state=42)

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=0.25, random_state=42
)

reg = LinearRegression()
reg.fit(Xr_train, yr_train)
yr_pred = reg.predict(Xr_test)

mse = mean_squared_error(yr_test, yr_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(yr_test, yr_pred)

print(f"MSE : {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE : {mae:.2f}")


## 10) Оптимизация: Gradient Descent (интуитивті түсіндіру)

Gradient Descent — Loss функциясын азайту үшін параметрлерді (w, b) біртіндеп жаңарту.

Мысал: 1D сызықтық модель:
\$
\hat{y} = w \cdot x + b
\$

Loss (MSE):
\$
L = \frac{1}{n}\sum (\hat{y} - y)^2
\$

Жаңарту:
- w := w − α · dL/dw  
- b := b − α · dL/db  

Мұндағы α — **learning rate** (қадам өлшемі).


In [ ]:
# 1D синтетикалық дерек
rng = np.random.RandomState(42)
x = rng.uniform(-3, 3, size=300)

true_w, true_b = 2.5, -1.0
y = true_w * x + true_b + rng.normal(0, 1.0, size=x.shape[0])

# Бастапқы параметрлер
w, b = 0.0, 0.0
lr = 0.05
epochs = 200

loss_history = []

for epoch in range(epochs):
    y_hat = w * x + b
    error = y_hat - y

    # MSE
    loss = np.mean(error**2)
    loss_history.append(loss)

    # Градиенттер (аналитикалық түрде)
    dL_dw = 2 * np.mean(error * x)
    dL_db = 2 * np.mean(error)

    # Параметрлерді жаңарту
    w -= lr * dL_dw
    b -= lr * dL_db

print(f"Learned w={w:.3f}, b={b:.3f}")
print(f"True    w={true_w:.3f}, b={true_b:.3f}")


In [ ]:
# Loss қалай төмендегенін көреміз
plt.figure()
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Gradient Descent — Loss төмендеуі")
plt.show()

# Модель сызығын салу
plt.figure()
plt.scatter(x, y, alpha=0.6, label="Data")
xs = np.linspace(x.min(), x.max(), 100)
plt.plot(xs, w*xs + b, linewidth=2, label="Fitted line (GD)")
plt.plot(xs, true_w*xs + true_b, linewidth=2, label="True line")
plt.legend()
plt.title("Gradient Descent — Нәтиже")
plt.show()


# Қорытынды (өте қысқа)

- **Accuracy** теңгерімсіз кластарда жиі жаңылыстырады → **Precision/Recall/F1** және **ROC-AUC** қараңыз.  
- **K-Fold CV** — әділ бағалау, нәтижені тұрақтандырады.  
- **GridSearch** толық, бірақ баяу; **RandomizedSearch** жиі жылдамырақ.  
- Imbalanced: **class_weight** және **SMOTE** — негізгі тәсілдер.  
- **Gradient Descent** — оптимизацияның базалық алгоритмі.
